In [6]:
import copy
import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import kagglehub
import matplotlib.image as mping
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import ConcatDataset, DataLoader, Dataset, WeightedRandomSampler, random_split

In [7]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

tushar5harma_plant_village_dataset_updated_path = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
shubham2703_five_crop_diseases_dataset_path = kagglehub.dataset_download('shubham2703/five-crop-diseases-dataset')
zunorain_pea_plant_dataset_path = kagglehub.dataset_download('zunorain/pea-plant-dataset')
nirmalsankalana_grape400_dataset_path = kagglehub.dataset_download('nirmalsankalana/grape400-dataset')
mexwell_crop_diseases_classification_path = kagglehub.dataset_download('mexwell/crop-diseases-classification')
ayushbpanchal_cashew_image_dataset_path = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')
tahmidmir_pumpkin_leaf_diseases_dataset_from_bangladesh_path = kagglehub.dataset_download('tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh')
fahadbd_plant_leaf_freshness_path = kagglehub.dataset_download('fahadbd/plant-leaf-freshness')
binnassor89_melon_macronutrient_deficiency_dataset_path = kagglehub.dataset_download('binnassor89/melon-macronutrient-deficiency-dataset')
asadullahgalib_guava_disease_dataset_path = kagglehub.dataset_download('asadullahgalib/guava-disease-dataset')
nirmalsankalana_rocole_a_robusta_coffee_leaf_images_dataset_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')
mahmoudshaheen1134_lemon_leaf_disease_dataset_lldd_path = kagglehub.dataset_download('mahmoudshaheen1134/lemon-leaf-disease-dataset-lldd')
shuvokumarbasak4004_flower_leaf_diseases_dataset_new_and_update_path = kagglehub.dataset_download('shuvokumarbasak4004/flower-leaf-diseases-dataset-new-and-update')
mihsanpermana_vanilla_plant_disease_image_dataset_path = kagglehub.dataset_download('mihsanpermana/vanilla-plant-disease-image-dataset')
tushar5harma_plant_village_dataset_updated_path = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
nirmalsankalana_rocole_a_robusta_coffee_leaf_images_dataset_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')
ayushbpanchal_cashew_image_dataset_path = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')


print('Data source import complete.')

Data source import complete.


# Dataset Configuration, Loading & SPlitting

In [8]:
BASE = "/kaggle/input/datasets"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── Complete label normalization maps (all 8 datasets) ─────────────────────
dataset_label_maps = {
    "shubham2703": {
        "Bacterial Blight"           : "Rice__Bacterial_Blight",
        "Corn___Common_Rust"         : "Corn__Common_Rust",
        "Corn___Gray_Leaf_Spot"      : "Corn__Gray_Leaf_Spot",
        "Corn___Healthy"             : "Corn__Healthy",
        "Corn___Northern_Leaf_Blight": "Corn__Northern_Leaf_Blight",
        "Healthy"                    : None,   # no crop prefix — skip
        "Potato___Early_Blight"      : "Potato__Early_Blight",
        "Potato___Healthy"           : "Potato__Healthy",
        "Potato___Late_Blight"       : "Potato__Late_Blight",
        "Red Rot"                    : "Sugarcane__Red_Rot",
        "Rice___Brown_Spot"          : "Rice__Brown_Spot",
        "Rice___Healthy"             : "Rice__Healthy",
        "Rice___Leaf_Blast"          : "Rice__Leaf_Blast",
        "Rice___Neck_Blast"          : "Rice__Neck_Blast",
        "Wheat___Brown_Rust"         : "Wheat__Brown_Rust",
        "Wheat___Healthy"            : "Wheat__Healthy",
        "Wheat___Yellow_Rust"        : "Wheat__Yellow_Rust",
    },
    "nirmalsankalana": {
        "BlackMeasles"               : "Grape__Black_Measles",
        "BlackRot"                   : "Grape__Black_Rot",
        "HealthyGrapes"              : "Grape__Healthy",
        "LeafBlight"                 : "Grape__Leaf_Blight",
        "coffee___healthy"           : "Coffee__Healthy",
        "coffee___red_spider_mite"   : "Coffee__Red_Spider_Mite",
        "coffee___rust"              : "Coffee__Rust",
    },
    "binnassor89": {
        "calcium"                    : "Melon__Calcium_Deficiency",
        "health"                     : "Melon__Healthy",
        "nitrogen"                   : "Melon__Nitrogen_Deficiency",
        "potasium"                   : "Melon__Potassium_Deficiency",
    },
    "zunorain": {
        "DOWNY_MILDEW_LEAF"          : "Pea__Downy_Mildew",
        "FRESH_LEAF"                 : "Pea__Healthy",
        "LEAFMINNER_LEAF"            : "Pea__Leaf_Miner",
        "POWDER_MILDEW_LEAF"         : "Pea__Powdery_Mildew",
    },
    "ayushbpanchal": {
        "anthracnose"                : "Cashew__Anthracnose",
        "gumosis"                    : "Cashew__Gummosis",
        "healthy"                    : "Cashew__Healthy",
        "leaf miner"                 : "Cashew__Leaf_Miner",
        "red rust"                   : "Cashew__Red_Rust",
    },
    "yasaminborhani": {
        "Brown_rust"                 : "Wheat__Brown_Rust",
        "Healthy"                    : "Wheat__Healthy",
        "Yellow_rust"                : "Wheat__Yellow_Rust",
    },
    "tahmidmir": {
        "Bacterial Leaf Spot"        : "Pumpkin__Bacterial_Leaf_Spot",
        "Downy Mildew"               : "Pumpkin__Downy_Mildew",
        "Healthy Leaf"               : "Pumpkin__Healthy",
        "Mosaic Disease"             : "Pumpkin__Mosaic_Disease",
        "Powdery_Mildew"             : "Pumpkin__Powdery_Mildew",
    },
    "mexwell": {
        "Cassava Bacterial Blight (CBB)"      : "Cassava__Bacterial_Blight",
        "Cassava Brown Streak Disease (CBSD)" : "Cassava__Brown_Streak_Disease",
        "Cassava Green Mottle (CGM)"          : "Cassava__Green_Mottle",
        "Cassava Mosaic Disease (CMD)"        : "Cassava__Mosaic_Disease",
        "Healthy"                             : "Cassava__Healthy",
    },
}

# ── tushar5harma label map ─────────────────────────────────────────────────
dataset_label_maps["tushar5harma"] = {
    # Apple
    ("Apple", "Apple Scab")              : "Apple__Apple_Scab",
    ("Apple", "Black Rot")               : "Apple__Black_Rot",
    ("Apple", "Cedar Apple Rust")        : "Apple__Cedar_Apple_Rust",
    ("Apple", "Healthy")                 : "Apple__Healthy",
    # Bell Pepper
    ("Bell Pepper", "Bacterial Spot")    : "Bell_Pepper__Bacterial_Spot",
    ("Bell Pepper", "Healthy")           : "Bell_Pepper__Healthy",
    # Cherry
    ("Cherry", "Healthy")                : "Cherry__Healthy",
    ("Cherry", "Powdery Mildew")         : "Cherry__Powdery_Mildew",
    # Corn — merge into Corn__ namespace (resolves (Maize) conflict)
    ("Corn (Maize)", "Cercospora Leaf Spot") : "Corn__Gray_Leaf_Spot",   # same disease
    ("Corn (Maize)", "Common Rust")          : "Corn__Common_Rust",
    ("Corn (Maize)", "Healthy")              : "Corn__Healthy",
    ("Corn (Maize)", "Northern Leaf Blight") : "Corn__Northern_Leaf_Blight",
    # Grape — merge into Grape__ namespace
    ("Grape", "Black Rot")               : "Grape__Black_Rot",
    ("Grape", "Esca (Black Measles)")    : "Grape__Black_Measles",       # merge with nirmalsankalana
    ("Grape", "Healthy")                 : "Grape__Healthy",
    ("Grape", "Leaf Blight")             : "Grape__Leaf_Blight",
    # Peach
    ("Peach", "Bacterial Spot")          : "Peach__Bacterial_Spot",
    ("Peach", "Healthy")                 : "Peach__Healthy",
    # Potato — already in namespace, will merge with shubham2703
    ("Potato", "Early Blight")           : "Potato__Early_Blight",
    ("Potato", "Healthy")                : "Potato__Healthy",
    ("Potato", "Late Blight")            : "Potato__Late_Blight",
    # Strawberry
    ("Strawberry", "Healthy")            : "Strawberry__Healthy",
    ("Strawberry", "Leaf Scorch")        : "Strawberry__Leaf_Scorch",
    # Tomato
    ("Tomato", "Bacterial Spot")         : "Tomato__Bacterial_Spot",
    ("Tomato", "Early Blight")           : "Tomato__Early_Blight",
    ("Tomato", "Healthy")                : "Tomato__Healthy",
    ("Tomato", "Late Blight")            : "Tomato__Late_Blight",
    ("Tomato", "Septoria Leaf Spot")     : "Tomato__Septoria_Leaf_Spot",
    ("Tomato", "Yellow Leaf Curl Virus") : "Tomato__Yellow_Leaf_Curl_Virus",
}

# Build global cannonical class -> index mapping
all_cannonical = sorted(set(
    label
    for mapping in dataset_label_maps.values()
    for label in mapping.values()
    if label is not None
))
class_to_idx = {cls: idx for idx, cls in enumerate(all_cannonical)}
n_classes = len(class_to_idx)
print(f"Total cannonical classes: {n_classes}")

Total cannonical classes: 64


# Data Augmentation

In [9]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dataset Class

In [10]:
class CropDiseaseDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label
    
def build_samples_folder(slug: str) -> list:
    label_map = dataset_label_maps[slug]
    root = Path(f"{BASE}/{slug}")
    samples = []
    skipped = 0

    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        folder_name = img_path.parent.name
        canonical = label_map.get(folder_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((img_path, class_to_idx[canonical]))

    print(f"[{slug}] {len(samples)} samples loaded,\t{skipped} skipped")
    return samples

def build_samples_mexwell() -> list:
    data_root = Path(f"{BASE}/mexwell/crop-diseases-classification/Data")
    img_dir   = data_root / "train_images"

    with open(data_root / "label_num_to_disease_map.json") as f:
        num_to_name = json.load(f)

    df = pd.read_csv(data_root / "train.csv")
    label_map = dataset_label_maps["mexwell"]
    samples = []
    skipped = 0

    for _, row in df.iterrows():
        img_path = img_dir / row["image_id"]
        if not img_path.exists():
            skipped += 1
            continue
        disease_name = num_to_name[str(row["label"])]
        canonical    = label_map.get(disease_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((img_path, class_to_idx[canonical]))

    print(f"  [mexwell] {len(samples)} samples loaded, {skipped} skipped")
    return samples

def build_samples_tushar() -> list:
    label_map = dataset_label_maps["tushar5harma"]
    root = Path(f"{BASE}/tushar5harma")
    samples = []
    skipped = 0

    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        rel = img_path.relative_to(root).parts
        if len(rel) < 4 or ".ipynb_checkpoints" in rel:
            skipped += 1
            continue
        crop    = rel[1].strip()
        disease = rel[3].strip()
        canonical = label_map.get((crop, disease))
        if canonical is None:
            skipped += 1
            continue
        samples.append((img_path, class_to_idx[canonical]))

    print(f"  [tushar5harma] {len(samples)} samples loaded, {skipped} skipped")
    return samples

# Building Samples

In [11]:
folder_slugs = [
    "shubham2703", "nirmalsankalana", "binnassor89",
    "zunorain", "ayushbpanchal", "yasaminborhani", "tahmidmir"
]

all_samples = []
for slug in folder_slugs:
    all_samples += build_samples_folder(slug)
all_samples += build_samples_mexwell()
all_samples += build_samples_tushar()

print(f"\nTotal samples before split: {len(all_samples)}")

val_size = int(0.15 * len(all_samples))
train_size = len(all_samples) - val_size

train_samples, val_samples = random_split(
    all_samples,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = CropDiseaseDataset(list(train_samples), transform=train_transform)
val_dataset   = CropDiseaseDataset(list(val_samples), transform=val_transform)

[shubham2703] 13224 samples loaded,	100 skipped
[nirmalsankalana] 3160 samples loaded,	0 skipped
[binnassor89] 200 samples loaded,	0 skipped
[zunorain] 1432 samples loaded,	0 skipped
[ayushbpanchal] 25811 samples loaded,	0 skipped
[yasaminborhani] 3679 samples loaded,	0 skipped
[tahmidmir] 2000 samples loaded,	0 skipped
  [mexwell] 17938 samples loaded, 3459 skipped
  [tushar5harma] 67111 samples loaded, 7 skipped

Total samples before split: 134555


# Training Process

In [12]:
os.makedirs("/kaggle/working/runs", exist_ok=True)

In [13]:
# ── Rebuild global class → index map (now includes tushar5harma) ───────────
all_canonical = sorted(set(
    label
    for mapping in dataset_label_maps.values()
    for label in mapping.values()
    if label is not None
))
class_to_idx = {cls: idx for idx, cls in enumerate(all_canonical)}
n_classes = len(class_to_idx)
print(f"Total canonical classes: {n_classes}")   # expect ~76

# ── Rebuild all_samples with tushar5harma ──────────────────────────────────
all_samples = []
for slug in folder_slugs:
    all_samples += build_samples_folder(slug)
all_samples += build_samples_mexwell()
all_samples += build_samples_tushar()          # ← new

print(f"\nTotal samples before split: {len(all_samples)}")

Total canonical classes: 64
[shubham2703] 13224 samples loaded,	100 skipped
[nirmalsankalana] 3160 samples loaded,	0 skipped
[binnassor89] 200 samples loaded,	0 skipped
[zunorain] 1432 samples loaded,	0 skipped
[ayushbpanchal] 25811 samples loaded,	0 skipped
[yasaminborhani] 3679 samples loaded,	0 skipped
[tahmidmir] 2000 samples loaded,	0 skipped
  [mexwell] 17938 samples loaded, 3459 skipped
  [tushar5harma] 67111 samples loaded, 7 skipped

Total samples before split: 134555


In [14]:
os.makedirs("/kaggle/working/runs", exist_ok=True)

# Handle Class Imbalance via DataLoaders
Configure class weighting for the `WeightedRandomSampler`. Then, build our standard PyTorch `DataLoader`s to pull structured batches optimally during training and evaluation loops.

In [15]:
label_counts = Counter(label for _, label in train_samples)
weights = [1.0 / label_counts[label] for _, label in train_samples]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

# Model Instantiation

In [16]:
PHASE1_WEIGHTS = "/kaggle/input/models/vitthaljauhari/resnet50-phase-1/pytorch/default/1/resnet50_phase1_best.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(2048, n_classes)
)
checkpoint = torch.load(PHASE1_WEIGHTS, map_location=device)
model.load_state_dict(checkpoint, strict=False)
model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

Using 2 GPUs


# Training & Evaluation Methods
Generate logic to selectively lock/freeze sub-layers, evaluate global validation scores (Top-1 & Top-5 accuracy constraints), and a comprehensive `train_phase` engine that iterates epochs and safely logs checkpoints.

In [17]:
def set_frozen(model, frozen_layers):
    for name, param in model.named_parameters():
        bare_name = name.replace("module.", "", 1)
        param.requires_grad = not any(bare_name.startswith(l) for l in frozen_layers)

def evaluate(model, loader):
    model.eval()
    correct_top1 = correct_top5 = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)

            preds = logits.argmax(dim=1)
            correct_top1 += (preds == labels).sum().item()

            top5 = logits.topk(5, dim=1).indices
            correct_top5 += sum(labels[i] in top5[i] for i in range(len(labels)))
            total += labels.size(0)
    return correct_top1 / total, correct_top5 / total

In [18]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# def train_phase(model, optimizer, scheduler, n_epochs, phase_name, best_acc, best_weights):
#     for epoch in range(1, n_epochs + 1):
#         model.train()
#         running_loss = correct = total = 0
#         t0 = time.time()

#         for imgs, labels in train_loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             optimizer.zero_grad()
#             logits = model(imgs)
#             loss = criterion(logits, labels)
#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item() * labels.size(0)
#             correct      += (logits.argmax(1) == labels).sum().item()
#             total        += labels.size(0)

#         scheduler.step()
        
#         train_loss = running_loss / total
#         train_acc  = correct / total
#         val_top1, val_top5 = evaluate(model, val_loader)
#         elapsed = time.time() - t0

#         print(f"[{phase_name}] Epoch {epoch} | "
#               f"loss {train_loss:.4f} | train {train_acc:.4f} | "
#               f"val_top1 {val_top1:.4f} | val_top5 {val_top5:.4f} | "
#               f"{elapsed:.0f}s")
        
#         if val_top1 > best_acc:
#             best_acc = val_top1
#             best_weights = copy.deepcopy(model.state_dict())
#             torch.save(best_weights, f"/kaggle/working/runs/phase2_best.pth")
#             print(f"New best saved ({best_acc:.4f})")

#     return best_acc, best_weights

def train_phase(model, optimizer, scheduler, n_epochs, phase_name, 
                best_acc=0.0, best_weights=None, start_epoch=0):

    for epoch in range(1, n_epochs + 1):
        global_epoch = start_epoch + epoch

        model.train()
        running_loss = correct = total = 0
        t0 = time.time()

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()

            logits = model(imgs)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        val_top1, val_top5 = evaluate(model, val_loader)

        # Step scheduler ONCE per epoch
        scheduler.step()

        elapsed = time.time() - t0

        print(
            f"[{phase_name}] Epoch {epoch} | "
            f"loss {train_loss:.4f} | train {train_acc:.4f} | "
            f"val_top1 {val_top1:.4f} | val_top5 {val_top5:.4f} | "
            f"{elapsed:.0f}s"
        )

        if val_top1 > best_acc:
            best_acc = val_top1

            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )

            torch.save({
                "phase": phase_name,
                "epoch": global_epoch,
                "model_state_dict": best_weights,
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_acc": best_acc,
            }, f"/kaggle/working/runs/{phase_name}_best.pth")

            print(f"New best saved ({best_acc:.4f})")

        torch.save({
            "phase": phase_name,
            "epoch": global_epoch,
            "model_state_dict": (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_acc": best_acc,
        }, f"/kaggle/working/runs/{phase_name}_epoch{global_epoch+1}.pth")

    return best_acc, best_weights

# Phase A Training Procedure
Execute the first brief fine-tuning regime where we clamp all base feature generators and optimize solely on our newly mapped FC layer. This prevents immediate corruption of learned feature spaces as it adapts to our 76-class setup.

In [22]:
# Phase A: fc only, 5 epochs
set_frozen(model, frozen_layers=["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"])

optimizer_A = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
scheduler_A = CosineAnnealingLR(optimizer_A, T_max=5, eta_min=1e-5)

best_acc, best_weights = train_phase(
    model, optimizer_A, scheduler_A,
    n_epochs=5, phase_name="A",
    best_acc=0.0, best_weights=None
)

[A] Epoch 1 | loss 1.8164 | train 0.6064 | val_top1 0.5811 | val_top5 0.9440 | 1143s
New best saved (0.5811)
[A] Epoch 2 | loss 1.4756 | train 0.7310 | val_top1 0.7640 | val_top5 0.9728 | 1143s
New best saved (0.7640)
[A] Epoch 3 | loss 1.3068 | train 0.7946 | val_top1 0.7749 | val_top5 0.9818 | 1116s
New best saved (0.7749)
[A] Epoch 4 | loss 1.1948 | train 0.8329 | val_top1 0.8314 | val_top5 0.9895 | 1115s
New best saved (0.8314)
[A] Epoch 5 | loss 1.1207 | train 0.8608 | val_top1 0.8292 | val_top5 0.9959 | 1136s


# Phase B Training Procedure
Relax the system selectively by activating `layer3`, `layer4` and the `fc` node entirely. We'll fine-tune deeper parameters natively to the dataset distributions.

In [19]:
phase1 = torch.load("/kaggle/input/models/vitthaljauhari/resnet50-phase-1/pytorch/default/1/resnet50_phase1_best.pth", map_location=device)
print(phase1.keys())

dict_keys(['epoch', 'model_state_dict', 'class_map', 'val_acc'])


In [20]:
# Phase B: unfreeze layer3+layer4+fc, 20 epochs
set_frozen(model, frozen_layers=["conv1", "bn1", "layer1", "layer2"])

m = model.module if isinstance(model, nn.DataParallel) else model

optimizer_B = torch.optim.Adam([
    {"params": m.layer3.parameters(), "lr": 5e-5},
    {"params": m.layer4.parameters(), "lr": 1e-4},
    {"params": m.fc.parameters(), "lr": 1e-4},
])

scheduler_B = CosineAnnealingLR(optimizer_B, T_max=20, eta_min=1e-6)

best_acc = 0.0
best_weights = None

best_acc, best_weights = train_phase(
    model, optimizer_B, scheduler_B,
    n_epochs=20, phase_name="B",
    best_acc=best_acc, best_weights=best_weights
)
print(f"\nPhase 2 training complete. Best val Top-1: {best_acc:.4f}")

[B] Epoch 0 | loss 3.0714 | train 0.2412 | val_top1 0.2412 | val_top5 0.6520 | 1436s
New best saved (0.2412)
[B] Epoch 1 | loss 2.3438 | train 0.4328 | val_top1 0.4495 | val_top5 0.8147 | 1414s
New best saved (0.4495)
[B] Epoch 2 | loss 2.0790 | train 0.5207 | val_top1 0.5054 | val_top5 0.8510 | 1405s
New best saved (0.5054)
[B] Epoch 3 | loss 1.8937 | train 0.5831 | val_top1 0.5646 | val_top5 0.9062 | 1442s
New best saved (0.5646)
[B] Epoch 4 | loss 1.7521 | train 0.6354 | val_top1 0.5947 | val_top5 0.9211 | 1414s
New best saved (0.5947)
[B] Epoch 5 | loss 1.6523 | train 0.6715 | val_top1 0.6554 | val_top5 0.9410 | 1430s
New best saved (0.6554)
[B] Epoch 6 | loss 1.5710 | train 0.7009 | val_top1 0.6736 | val_top5 0.9587 | 1426s
New best saved (0.6736)
[B] Epoch 7 | loss 1.5119 | train 0.7216 | val_top1 0.6915 | val_top5 0.9617 | 1487s
New best saved (0.6915)
[B] Epoch 8 | loss 1.4548 | train 0.7441 | val_top1 0.7303 | val_top5 0.9691 | 1414s
New best saved (0.7303)
[B] Epoch 9 | loss 

In [23]:
m = model.module if isinstance(model, nn.DataParallel) else model

checkpoint = {
    "epoch": 20,
    "phase": "B",
    "model_state_dict": m.state_dict(),
    "optimizer_state_dict": optimizer_B.state_dict(),
    "scheduler_state_dict": scheduler_B.state_dict(),
    "best_acc": best_acc,
    "classes_to_idx": class_to_idx,
    "val_top1": 0.8072,
    "val_top5": 0.9876,
    "n_classes": n_classes,
}

save_path = "/kaggle/working/runs/phase2_best.pth"
torch.save(checkpoint, save_path)
print(f"Saved to {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1e6:.1f} MB")

Saved to /kaggle/working/runs/phase2_best.pth
File size: 272.5 MB


In [26]:
# Verify
ckpt = torch.load("/kaggle/working/runs/phase2_best.pth", map_location=device)
print("Keys     :", list(ckpt.keys()))
print("Phase    :", ckpt["phase"])
print("Epoch    :", ckpt["epoch"])
print("Best acc :", ckpt["best_acc"])
print("Classes  :", ckpt["n_classes"])

m_test = models.resnet50(weights=None)
m_test.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, ckpt["n_classes"]))
m_test.load_state_dict(ckpt["model_state_dict"])

Keys     : ['epoch', 'phase', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'best_acc', 'classes_to_idx', 'val_top1', 'val_top5', 'n_classes']
Phase    : B
Epoch    : 20
Best acc : 0.8071644453252738
Classes  : 64


<All keys matched successfully>

In [ ]:
RESUME_PATH = "/kaggle/input/.../B_epoch8.pth"

ckpt = torch.load(RESUME_PATH, map_location=device)

m = model.module if isinstance(model, nn.DataParallel) else model

m.load_state_dict(
    ckpt["model_state_dict"], strict=True
)

# Rebuild optimizer with same param groups
optimizer_B = torch.optim.Adam([
    {"params": m.layer3.parameters(), "lr": 5e-5},
    {"params": m.layer4.parameters(), "lr": 1e-4},
    {"params": m.fc.parameters(),     "lr": 1e-4},
])
scheduler_B = CosineAnnealingLR(optimizer_B, T_max=20, eta_min=1e-6)

# Restore full state — no manual fast-forward, no cold Adam
optimizer_B.load_state_dict(ckpt["optimizer_state_dict"])
scheduler_B.load_state_dict(ckpt["scheduler_state_dict"])

EPOCHS_DONE = ckpt["epoch"] + 1  # auto from checkpoint
BEST_ACC    = ckpt["best_acc"]

best_acc, best_weights = train_phase(
    model, optimizer_B, scheduler_B,
    n_epochs=20 - EPOCHS_DONE,
    phase_name="B",
    best_acc=BEST_ACC,
    start_epoch=EPOCHS_DONE,
)

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[B-resume] Epoch 1 | loss 3.2807 | train 0.1707 | val_top1 0.2317 | val_top5 0.5819 | 887s
[B-resume] Epoch 2 | loss 2.5934 | train 0.3394 | val_top1 0.3033 | val_top5 0.7172 | 871s
[B-resume] Epoch 3 | loss 2.3762 | train 0.4044 | val_top1 0.3664 | val_top5 0.7471 | 867s
[B-resume] Epoch 4 | loss 2.2329 | train 0.4503 | val_top1 0.4183 | val_top5 0.8211 | 861s


KeyboardInterrupt: 